In [9]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, random_split
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
from collections import Counter
import numpy as np
from tqdm.auto import tqdm


# --- Config ---
DATA_DIR = "/kaggle/input/datasets/adityasharma01/snake-dataset-india/Snake Images/train"
BATCH_SIZE = 32
EPOCHS = 30
VAL_SPLIT = 0.2
EARLY_STOPPING_PATIENCE = 5
FREEZE_EPOCHS = 3  # train only classifier first

# --- Transforms ---
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(0.2, 0.2, 0.2, 0.1),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# --- Dataset ---
dataset = datasets.ImageFolder(DATA_DIR, transform=transform)

# --- Compute class weights ---
labels = [label for _, label in dataset.samples]
counts = Counter(labels)
total = sum(counts.values())
weights = [total / counts[i] for i in range(len(counts))]
class_weights = torch.tensor(weights)

# Split dataset
val_size = int(len(dataset) * VAL_SPLIT)
train_size = len(dataset) - val_size
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

# --- Model ---
model = models.efficientnet_b0(weights="IMAGENET1K_V1")
model.classifier[1] = nn.Linear(model.classifier[1].in_features, 2)

# Freeze backbone initially
for param in model.features.parameters():
    param.requires_grad = False

# --- Device ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
class_weights = class_weights.to(device)

# --- Loss & Optimizer ---
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=3e-4)

# --- Scheduler ---
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', patience=2, factor=0.3
)

# --- Early Stopping ---
best_val_loss = float("inf")
patience_counter = 0
# --- Training Loop ---
for epoch in range(EPOCHS):

    if epoch == FREEZE_EPOCHS:
        print("Unfreezing backbone...")
        for param in model.features.parameters():
            param.requires_grad = True
        optimizer = optim.Adam(model.parameters(), lr=1e-4)

    # --- Train ---
    model.train()
    train_loss = 0

    train_bar = tqdm(train_loader, desc=f"Epoch {epoch+1} [Train]", leave=False)

    for images, labels in train_bar:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

        train_bar.set_postfix(loss=loss.item())

    # --- Validation ---
    model.eval()
    val_loss = 0
    all_preds = []
    all_labels = []

    val_bar = tqdm(val_loader, desc=f"Epoch {epoch+1} [Val]", leave=False)

    with torch.no_grad():
        for images, labels in val_bar:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item()

            preds = torch.argmax(outputs, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

            val_bar.set_postfix(loss=loss.item())

    
    avg_train_loss = train_loss / len(train_loader)
    avg_val_loss = val_loss / len(val_loader)

    # Metrics
    accuracy = accuracy_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds, average='binary')
    recall = recall_score(all_labels, all_preds, average='binary')
    f1 = f1_score(all_labels, all_preds, average='binary')

    print(f"\nEpoch {epoch+1}")
    print(f"Train Loss: {avg_train_loss:.4f}")
    print(f"Val Loss  : {avg_val_loss:.4f}")
    print(f"F1        : {f1:.4f} | Precision: {precision:.4f} | Recall: {recall:.4f}")

    # Step scheduler
    scheduler.step(avg_val_loss)
    current_lr = optimizer.param_groups[0]['lr']
    print(f"Current LR: {current_lr:.6f}")

    # --- Early Stopping ---
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        patience_counter = 0
        torch.save(model.state_dict(), "/kaggle/working/venom.pt")
    else:
        patience_counter += 1

    if patience_counter >= EARLY_STOPPING_PATIENCE:
        print("Early stopping triggered.")
        break

# --- Final Evaluation ---
cm = confusion_matrix(all_labels, all_preds)
print("\nConfusion Matrix:")
print(cm)

Epoch 1 [Train]:   0%|          | 0/45 [00:00<?, ?it/s]

Epoch 1 [Val]:   0%|          | 0/12 [00:00<?, ?it/s]


Epoch 1
Train Loss: 0.6880
Val Loss  : 0.6463
F1        : 0.6718 | Precision: 0.7021 | Recall: 0.6439
Current LR: 0.000300


Epoch 2 [Train]:   0%|          | 0/45 [00:00<?, ?it/s]

Epoch 2 [Val]:   0%|          | 0/12 [00:00<?, ?it/s]

...

Epoch 20 [Val]:   0%|          | 0/12 [00:00<?, ?it/s]


Epoch 20
Train Loss: 0.0335
Val Loss  : 0.2223
F1        : 0.9185 | Precision: 0.9300 | Recall: 0.9073
Current LR: 0.000100


Epoch 21 [Train]:   0%|          | 0/45 [00:00<?, ?it/s]

Epoch 21 [Val]:   0%|          | 0/12 [00:00<?, ?it/s]


Epoch 21
Train Loss: 0.0301
Val Loss  : 0.2777
F1        : 0.9100 | Precision: 0.9078 | Recall: 0.9122
Current LR: 0.000100
Early stopping triggered.

Confusion Matrix:
[[131  19]
 [ 18 187]]
